# TinyForge — d6 and d12 on ClimbMix

Trains the modern TinyForge decoder (RoPE, parameter-free RMSNorm, QK-norm, ReLU², untied head, logit softcap, Muon) from scratch on one GPU. The stock llm.c GPT-2 baseline stays available as `third_party/llm.c/train_gpt2cu` and is not this notebook's default.

Runtime disk holds the token shards, checkpoints, and logs. The only things copied to Google Drive are the Hugging Face folders `MyDrive/llm.c/models/d6-hf` and `MyDrive/llm.c/models/d12-hf` (the files sit directly in those folders).

Micro-batch is `-b 16`. Total batch is 262144 tokens via gradient accumulation. Do not raise the micro-batch to 32; that OOMs.

After this branch is merged, set `TINYFORGE_BRANCH` to `main` in the setup cell.


In [ ]:
import os
from pathlib import Path

BRANCH = os.environ.get("TINYFORGE_BRANCH", "cursor/tinyforge-llmc-8a93")
REPO = os.environ.get("TINYFORGE_REPO", "https://github.com/michaelp91-dev/tinyforge.git")

if not Path("src/train_tinyforge.cu").exists():
    %cd /content
    !rm -rf /content/tinyforge
    !git clone --branch {BRANCH} {REPO} /content/tinyforge
    %cd /content/tinyforge

!nvidia-smi -L
!python -m pip install -q torch numpy tokenizers transformers datasets pyarrow requests tiktoken
print("cwd", Path(".").resolve())


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/llm.c/models"
os.makedirs(DRIVE, exist_ok=True)
print("Drive exports go to", DRIVE)


## Build

`train_tinyforge` is the modern trainer. `train_gpt2cu` is unmodified llm.c (plus the CUDA 13 `cudaMemAdvise` fix already in the vendor tree) and is only needed if you want to reproduce the stock GPT-2 bpb numbers.


In [ ]:
!make train_tinyforge
!make -C third_party/llm.c train_gpt2cu
!ls -l train_tinyforge third_party/llm.c/train_gpt2cu


## Data

Byte-level BPE, vocab 32768, trained on ClimbMix, with `<|bos|>` in front of every document. Shards are llm.c `uint16` files on the runtime disk. 1.4B train tokens covers the d12 run (5040 steps × 262144) and the shorter d6 run.

Tokenizer training reads 500M characters. Raise `--max-chars` if you want a closer match to nanochat's 2B-character tokenizer.


In [ ]:
import os
os.makedirs("/content/data", exist_ok=True)
!python python/train_tokenizer.py --out /content/data/tokenizer --vocab-size 32768 --max-chars 500000000
!python python/prepare_climbmix.py --tokenizer /content/data/tokenizer --out /content/data/climbmix --target-tokens 1400000000 --val-tokens 262144
!ls -lh /content/data/climbmix | head


## d6

Depth 6, width 384, 3 heads. Scaling params (matrices + untied head) are about 23.2M, so 12 tokens per param is 1062 steps at batch 262144. Checkpoints go to `/content/d6`, which is the Colab disk, not Drive.


In [ ]:
%%bash
mkdir -p /content/d6
./train_tinyforge \
  -i "/content/data/climbmix/train_*.bin" -j /content/data/climbmix/val.bin \
  -e d6 -o /content/d6 \
  -b 16 -t 1024 -d 262144 -x 1062 \
  -v 250 -n 250 -y 1 \
  --vocab 32768 \
  --token-bytes /content/data/tokenizer/token_bytes.bin


## d12

Depth 12, width 768, 6 heads. About 110.1M scaling params → 5040 steps. Same micro-batch and total batch. `-y 1` only resumes if `/content/d12` already has a checkpoint; a fresh directory starts from scratch.


In [ ]:
%%bash
mkdir -p /content/d12
./train_tinyforge \
  -i "/content/data/climbmix/train_*.bin" -j /content/data/climbmix/val.bin \
  -e d12 -o /content/d12 \
  -b 16 -t 1024 -d 262144 -x 5040 \
  -v 250 -n 250 -y 1 \
  --vocab 32768 \
  --token-bytes /content/data/tokenizer/token_bytes.bin


## Export

Latest checkpoint in each run directory becomes `/content/d6-hf` and `/content/d12-hf`, then those two folders are copied to Drive. Nothing else is uploaded.


In [ ]:
import os, shutil
from pathlib import Path

def latest_model(run_dir):
    dones = list(Path(run_dir).glob("DONE_*"))
    if not dones:
        raise SystemExit(f"no checkpoint in {run_dir}")
    step = max(int(p.name.split("_")[1]) for p in dones)
    return Path(run_dir) / f"model_{step:08d}.bin", step

def export_one(name):
    ckpt, step = latest_model(f"/content/{name}")
    local = Path(f"/content/{name}-hf")
    if local.exists():
        shutil.rmtree(local)
    print(f"exporting {ckpt}")
    !python python/export_hf.py --input {ckpt} --output {local} --tokenizer /content/data/tokenizer
    dest = Path(DRIVE) / f"{name}-hf"
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(local, dest)
    print("copied", dest)

export_one("d6")
export_one("d12")
print("Drive now has:")
for p in sorted(Path(DRIVE).glob("*-hf")):
    print(" ", p)


## Score

Official number is bits per byte on ClimbMix `shard_06542`, not the trainer's `val.bin` loss. This needs the GPU runtime. nanochat D6 on this eval was 0.9938. Stock llm.c d6 was 1.2110 at ~470M tokens and 1.1037 at 1.2B; llm.c d12 was 1.0115. The modern run has to be measured here; it was not measured when the code was written.


In [ ]:
import os
os.environ["MODEL_DIR"] = "/content/d6-hf"
os.environ["EVAL_TOKENS"] = str(80 * 524288)
os.environ["B"] = "16"
!python python/eval_bpb.py

os.environ["MODEL_DIR"] = "/content/d12-hf"
!python python/eval_bpb.py


## Stock llm.c baseline (optional)

Same data layout, GPT-2 tokenizer, vendored `train_gpt2cu`. This is the architecture behind the 1.21 / 1.10 / 1.01 numbers, not the modern decoder. It still uses `-b 16`.

```bash
python python/prepare_climbmix.py --tokenizer gpt2 --out /content/data/climbmix-gpt2 --target-tokens 1400000000
./third_party/llm.c/train_gpt2cu -i "/content/data/climbmix-gpt2/train_*.bin" -j /content/data/climbmix-gpt2/val.bin \
    -e d12 -b 16 -t 1024 -d 262144 -x 5040 -o /content/llm.c-d12
```
